In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.LevelSetTools.PhasefieldLevelSet;

Init();

This notebook starts simulations of the rising bubble benchmark test case 2 (https://doi.org/10.1002/fld.1934).  
The simulations are run with the cahn hilliard equation as a level advection equation.  
Different mobility parameters are used to highlight the impact the addtional diffusion in the c-h equation has on the interface behavior.

In [ ]:
string proj = "PhasefieldRisingBubble";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();


In [ ]:
static XNSE_Control RB_BenchmarkTest(int p = 2, int kelem = 10, int amrlevel = 2, int testcase = 2, LevelSetEvolution evo = LevelSetEvolution.FastMarching, double diff = 0.1, bool masscorrection = false) {

    XNSE_Control C = new XNSE_Control();

    // basic database options
    // ======================
    C.savetodb = true;
    C.SetDatabase(BoSSSshell.WorkflowMgm.DefaultDatabase);
    C.ProjectName = "PhasefieldRisingBubble";
    C.ProjectDescription = "rising bubble";
    C.Tags.Add("benchmark setup");

    C.PostprocessingModules.Add(new BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases.RisingBubble2DBenchmarkQuantities());

    // DG degrees
    // ==========
    C.SetDGdegree(p);

    // Physical Parameters
    // ===================
    switch(testcase){
        case 1:
            C.Tags.Add("Testcase 1");
            C.PhysicalParameters.rho_A = 100;
            C.PhysicalParameters.rho_B = 1000;
            C.PhysicalParameters.mu_A = 1;
            C.PhysicalParameters.mu_B = 10;
            C.PhysicalParameters.Sigma = 24.5;
            break;
        case 2:
            C.Tags.Add("Testcase 2");
            C.PhysicalParameters.rho_A = 1;
            C.PhysicalParameters.rho_B = 1000;
            C.PhysicalParameters.mu_A = 0.1;
            C.PhysicalParameters.mu_B = 10;
            C.PhysicalParameters.Sigma = 1.96;
            break;
        default:
            throw new NotImplementedException();
    }

    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = true;

    // grid generation
    // ===============

    double h = 1.0 / (double)kelem;
    C.GridFunc = delegate () {
        double[] Xnodes = GenericBlas.Linspace(0, 1.0, kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(0, 2.0, 2 * kelem + 1);
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, periodicX: false);

        grd.EdgeTagNames.Add(1, "wall_lower");
        grd.EdgeTagNames.Add(2, "wall_upper");
        grd.EdgeTagNames.Add(3, "freeslip_left");
        grd.EdgeTagNames.Add(4, "freeslip_right");

        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 0;
            if(Math.Abs(X[1]) <= 1.0e-8)
                et = 1;
            if(Math.Abs(X[1] - 2.0) <= 1.0e-8)
                et = 2;
            if(Math.Abs(X[0]) <= 1.0e-8)
                et = 3;
            if(Math.Abs(X[0] - 1.0) <= 1.0e-8)
                et = 4;
            return et;
        });

        grd.AddPredefinedPartitioning("TwoProcSplit", delegate (double[] X)
        {
            int rank;
            double x = X[0];
            if (x < 0.5)
                rank = 0;
            else 
                rank = 1;

            return rank;
        });

        return grd;
    };

    C.GridPartType = GridPartType.Predefined;
    C.GridPartOptions = "TwoProcSplit";

    // Initial Values
    // ==============
    double[] center = new double[] { 0.5, 0.5 };
    double radius = 0.25;    
    
    C.Option_LevelSetEvolution = evo;
    C.Tags.Add(evo.ToString());
    Formula PhiFunc;
    if(evo == LevelSetEvolution.Phasefield){
        double cahn = 4 * 1.0/(kelem * Math.Pow(2.0, amrlevel)) / (2 * p + 1);
        PhiFunc = new Formula($"X => Math.Tanh((((X[0] - {center[0]}).Pow2() + (X[1] - {center[1]}).Pow2()).Sqrt() - {radius})/(Math.Sqrt(2)*{cahn}))"); // signed-distance form
        C.PhasefieldControl = new PhasefieldSettings() {
            cahn = cahn,
            diff = diff,
            TimeSteppingScheme = TimeSteppingScheme.SDIRK_22,
            CorrectionType = masscorrection? BoSSS.Solution.LevelSetTools.PhasefieldLevelSet.Correction.Mass : BoSSS.Solution.LevelSetTools.PhasefieldLevelSet.Correction.None
        };
    }else{
        PhiFunc = new Formula($"X => (X[0] - {center[0]}).Pow2() + (X[1] - {center[1]}).Pow2() - {radius}.Pow2()"); // quadratic form
    }
       
    C.AddInitialValue("Phi", PhiFunc);

    C.AddInitialValue("VelocityX#A", new Formula("X => 0.0"));
    C.AddInitialValue("VelocityX#B", new Formula("X => 0.0"));

    C.AddInitialValue("GravityY#A", new Formula("X => -9.81e-1"));
    C.AddInitialValue("GravityY#B", new Formula("X => -9.81e-1"));

    // boundary conditions
    // ===================
    C.AddBoundaryValue("wall_lower");
    C.AddBoundaryValue("wall_upper");
    C.AddBoundaryValue("freeslip_left");
    C.AddBoundaryValue("freeslip_right");

    // misc. solver options
    // ====================
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

    C.NonLinearSolver.MaxSolverIterations = 20;
    C.FailOnSolverFail = false;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-6;

    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

    C.AdaptiveMeshRefinement = amrlevel > 0;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = amrlevel });
    C.AMR_startUpSweeps = amrlevel + 1;

    // Timestepping
    // ============
    C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
    C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;

    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;

    double dt = 0.001;
    C.dtMax = dt;
    C.dtMin = dt;
    C.NoOfTimesteps = (int)Math.Ceiling(3.0/dt);
    C.Endtime = 3.0;
    C.SuperSampling = 1;
    C.saveperiod = 10;
    C.rollingSaves = true;
    
    C.SessionName = "RisingBubble_" + testcase + "_AMR" + amrlevel + "_P" + p + "_" + evo.ToString() + (evo == LevelSetEvolution.Phasefield ? "_" + C.PhasefieldControl.CorrectionType.ToString() + "_" + C.PhasefieldControl.diff.ToString("N4") : "");
    // C.TracingNamespaces = "BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater";
    //C.TracingNamespaces = "*";
    return C;
}

In [ ]:
List<XNSE_Control> controls = new List<XNSE_Control>();

In [ ]:
//controls.Add(RB_BenchmarkTest(evo:LevelSetEvolution.FastMarching));
//controls.Add(RB_BenchmarkTest(evo:LevelSetEvolution.StokesExtension));
controls.Add(RB_BenchmarkTest(evo:LevelSetEvolution.Phasefield, diff: 1.0));
controls.Add(RB_BenchmarkTest(evo:LevelSetEvolution.Phasefield, diff: 0.1));
controls.Add(RB_BenchmarkTest(evo:LevelSetEvolution.Phasefield, diff: 0.01));
controls.Add(RB_BenchmarkTest(evo:LevelSetEvolution.Phasefield, diff: 0.001));
controls.Add(RB_BenchmarkTest(evo:LevelSetEvolution.Phasefield, diff: 0.0001));
//controls.Add(RB_BenchmarkTest(evo:LevelSetEvolution.Phasefield, diff: 0.00001));

//controls.Add(RB_BenchmarkTest(p: 3, evo:LevelSetEvolution.Phasefield, diff: 0.0001));
//controls.Add(RB_BenchmarkTest(amrlevel: 3, evo:LevelSetEvolution.Phasefield, diff: 0.0001));
//controls.Add(RB_BenchmarkTest(evo:LevelSetEvolution.Phasefield, diff: 0.0001, masscorrection: true));

int ctrlCount = controls.Count();

In [ ]:
// // FOR TESTING IF THIS RUNS

// var C = controls.First();
// //C.ImmediatePlotPeriod = 1;
// //C.SuperSampling = 2;
// //C.savetodb = false;
// //C.PostprocessingModules.Clear();
// using(var solver = new XNSE()){
//     solver.Init(C);
//     solver.RunSolverMode();
// }



In [ ]:
bool run      = true;
var bpc = BoSSSshell.GetDefaultQueue();
bpc.Display();

In [ ]:
//BoSSSshell.WorkflowMgm.ResetProject(true, true, true, true);

In [ ]:
if(BoSSSshell.WorkflowMgm.AllJobs.Count() > 0){
     BoSSSshell.WorkflowMgm.ResetProject();
}
var jobs = controls.Select(c => c.CreateJob()).ToArray();
jobs.ForEach(j => j.NumberOfMPIProcs = 2);
jobs.ForEach(j => j.NumberOfThreads = 1);
jobs.ForEach(j => j.EnvironmentVars["BOSSS_ARG_7"] = j.NumberOfThreads.ToString());

In [ ]:
jobs.Activate();

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(3*86400);

In [ ]:
int count = BoSSSshell.wmg.Sessions.Count();
int success = BoSSSshell.wmg.Sessions.Where(s => s.SuccessfulTermination).Count();

if(count != ctrlCount || count != success){
    throw new ApplicationException("Not all simulations calculated or finished successful!");
}

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions.ForEach(s => display(s));

In [ ]:
sessions.Count()

In [ ]:
// Für Vislt notwendig

//sessions.Skip(0).Take(1).ForEach(session=> session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).Do());
// session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).WithShadowFields().Do();
// session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).Do();

In [ ]:
// Process.Start("explorer.exe", sessions.Third().GetSessionDirectory());